1. Imports + env

In [2]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Notebook location: api/notebooks
API_ROOT = Path("..").resolve()                 # .../ai-tv-pro/api
TV_DATA_DIR = (API_ROOT / "tv_data").resolve() # .../ai-tv-pro/api/tv_data

# Load environment variables
load_dotenv(API_ROOT / ".env")

# Allow imports from api/
sys.path.insert(0, str(API_ROOT))

print("API_ROOT:", API_ROOT)
print("TV_DATA_DIR:", TV_DATA_DIR)
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))

API_ROOT: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api
TV_DATA_DIR: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data
OPENAI_API_KEY present: True


2. Load testset_tv.json

In [3]:
import pandas as pd

TESTSET_PATH = TV_DATA_DIR / "testset_tv.json"

test_df = pd.read_json(TESTSET_PATH)

print(f"Loaded {len(test_df)} test samples from: {TESTSET_PATH.resolve()}")
print("Columns:", list(test_df.columns))

test_df.head(3)

Loaded 27 test samples from: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\testset_tv.json
Columns: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,In the context of the IPTV Monthly Report for ...,[IPTV Monthly Report for month_partition=20241...,"In the IPTV Monthly Report for December 2024, ...",Customer Care Lead,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What are the viewer statistics for Doma TV ove...,[- channel=HRT 1 | playback=LTV | minutes=3372...,"Doma TV had a total of 6,326,153 minutes of pl...",Customer Care Lead,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What top channel in IPTV Monthly Report?,[IPTV Monthly Report for month_partition=20250...,Top entry is Nova TV (LTV) with 51622038 minut...,Customer Care Lead,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


3. Evaluation agent

In [7]:
from langchain.agents import create_agent
from app.agent import tv_rag_search, SYSTEM_PROMPT

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[tv_rag_search],
    system_prompt=SYSTEM_PROMPT,
)

print("Evaluation agent initialized successfully")
print("Tool:", tv_rag_search.name if hasattr(tv_rag_search, "name") else type(tv_rag_search))

Evaluation agent initialized successfully
Tool: tv_rag_search


4. Run agent on testset

In [9]:
from typing import Any, List

# Pretvaram rezultat retrievala u listu stringova
def to_context_list(x: Any) -> List[str]:
    if x is None:
        return []
    # ako imam LangChain Document
    if isinstance(x, list) and len(x) > 0 and hasattr(x[0], "page_content"):
        return [d.page_content for d in x]
    # ako imam lista stringova ili dictova
    if isinstance(x, list):
        return [str(i) for i in x]
    return [str(x)]

dataset = []

for i, row in test_df.iterrows():
    question = row["user_input"]

    # Progres
    print(f"[{i+1}/{len(test_df)}] {question[:100]}")

    # 1) retrieval preko tv_rag_search
    try:
        retrieved = tv_rag_search.invoke({"query": question})
    except Exception:
        # fallback ako tool prima samo string
        retrieved = tv_rag_search.invoke(question)

    retrieved_contexts = to_context_list(retrieved)

    # 2) agent odgovor
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": question}]}
    )

    if isinstance(result, dict) and "messages" in result and len(result["messages"]) > 0:
        response = result["messages"][-1].content
    else:
        response = str(result)

    dataset.append(
        {
            "user_input": question,
            "retrieved_contexts": retrieved_contexts,
            "response": response,
            "reference": row.get("reference", ""),
        }
    )

print(f"\nCollected {len(dataset)} samples")

[1/27] In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in te
[2/27] What are the viewer statistics for Doma TV over the last 31 days?
[3/27] What top channel in IPTV Monthly Report?
[4/27] How does RTL perform in terms of viewer engagement compared to other channels?
[5/27] What is the LTV and how many minutes and viewers MAXSport 1 got in the report?
[6/27] How many viewing minutes and viewers did STAR Movies have in the last 28 days?
[7/27] Wht are the total minits and unique viwers for HRT 1?
[8/27] How does the viewership of the STAR Crime channel compare to other channels in terms of total minute
[9/27] What top channel in IPTV report have most minutes and viewers?
[10/27] What are the top channel statistics for RTL in the IPTV Monthly Report for January 2026 compared to 
[11/27] Wich channel had the most viewers in the IPTV Monthly Report for 202509 and how does it compare to t
[12/27] In the IPTV Monthly Report for January 2026, how

5. RAGAS workaround

In [18]:
import json, os, sys, subprocess, textwrap
from pathlib import Path

def evaluate_ragas_subprocess(dataset_list, out_path: Path, timeout_s: int = 300):
    """
    Runs RAGAS evaluate() in a separate Python process and returns (stdout, stderr, returncode).
    Writes result summary to out_path.
    """
    TV_DATA_DIR.mkdir(parents=True, exist_ok=True)

    in_path = TV_DATA_DIR / "_eval_input.json"
    runner_path = TV_DATA_DIR / "_ragas_eval_runner.py"

    # input json
    in_path.write_text(json.dumps(dataset_list, ensure_ascii=False), encoding="utf-8")

    runner_code = textwrap.dedent(f"""
    import os, json
    from pathlib import Path
    from openai import OpenAI

    from ragas import EvaluationDataset, evaluate
    from ragas.run_config import RunConfig
    from ragas.llms import llm_factory
    from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

    IN_PATH = Path(os.environ["EVAL_INPUT"])
    OUT_PATH = Path(os.environ["EVAL_OUTPUT"])

    data = json.loads(IN_PATH.read_text(encoding="utf-8"))
    eval_dataset = EvaluationDataset.from_list(data)

    metrics = [LLMContextRecall(), Faithfulness(), FactualCorrectness()]

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    llm = llm_factory("gpt-4o-mini", client=client)

    run_config = RunConfig(timeout={timeout_s}, max_workers=1)

    result = evaluate(dataset=eval_dataset, metrics=metrics, llm=llm, run_config=run_config)

    OUT_PATH.write_text(str(result), encoding="utf-8")
    print(result)
    """).strip()

    runner_path.write_text(runner_code, encoding="utf-8")

    env = os.environ.copy()
    env["EVAL_INPUT"] = str(in_path)
    env["EVAL_OUTPUT"] = str(out_path)

    completed = subprocess.run(
        [sys.executable, str(runner_path)],
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    return completed.stdout, completed.stderr, completed.returncode

6. Evaluate with RAGAS

In [19]:
EVAL_OUT_PATH = TV_DATA_DIR / "eval_results_tv.txt"

stdout, stderr, rc = evaluate_ragas_subprocess(
    dataset,
    out_path=EVAL_OUT_PATH,
    timeout_s=300,
)

print("Return code:", rc)
print("Result (stdout):\n", stdout)

if stderr and stderr.strip():
    print("\nWarnings/Errors (stderr):\n", stderr)

print("\nSaved to:", EVAL_OUT_PATH)

Return code: 0
Result (stdout):
 {'context_recall': 0.0000, 'faithfulness': 0.0541, 'factual_correctness(mode=f1)': 0.0870}


Warnings/Errors (stderr):
 c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\_ragas_eval_runner.py:8: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
C:\Users\gpudja\OneDrive - Hrvatski Telekom

7. SQL correctnes

In [28]:
import duckdb
import pandas as pd
from pathlib import Path

CSV_PATH = (TV_DATA_DIR / "Monthly iptv - channel monthly rating.csv").resolve()
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

con = duckdb.connect(database=":memory:")

con.execute(f"""
    CREATE OR REPLACE VIEW tv AS
    SELECT
        month_partition::VARCHAR AS month_partition,
        CAST(watch_date AS DATE) AS watch_date,
        channelname::VARCHAR AS channelname,
        content_playback_type::VARCHAR AS content_playback_type,
        CAST(REPLACE(CAST(daily_total_minute AS VARCHAR), ',', '.') AS DOUBLE) AS daily_total_minute,
        CAST(REPLACE(CAST(nr_unique_viewers AS VARCHAR), ',', '.') AS DOUBLE) AS nr_unique_viewers
    FROM read_csv_auto('{CSV_PATH.as_posix()}', delim=';', header=True)
""")

print("tv view ready ✅")
print("Sample:", con.execute("SELECT daily_total_minute, nr_unique_viewers FROM tv LIMIT 5").fetchall())

tv view ready ✅
Sample: [(163974.13, 4413.0), (29304.38, 3768.0), (26797.96, 509.0), (229392.07, 9235.0), (4724.54, 243.0)]


8. SQL correctness - last 3 mnth

In [29]:
# 1) Dohvati zadnja 3 mjeseca (month_partition je 'YYYYMM' string)
last3 = con.execute("""
    SELECT month_partition
    FROM (SELECT DISTINCT month_partition FROM tv)
    ORDER BY month_partition DESC
    LIMIT 3
""").fetchall()

last3_months = [r[0] for r in last3]
print("Last 3 month_partitions:", last3_months)

# 2) Napravi view samo za ta 3 mjeseca
# (duckdb param binding za IN list je nezgrapan, pa radimo safe string list)
months_sql = ", ".join([f"'{m}'" for m in last3_months])

con.execute(f"""
    CREATE OR REPLACE VIEW tv_last3m AS
    SELECT *
    FROM tv
    WHERE month_partition IN ({months_sql})
""")

print("Rows in tv_last3m:", con.execute("SELECT COUNT(*) FROM tv_last3m").fetchone()[0])
print("Date range last3m:", con.execute("SELECT MIN(watch_date), MAX(watch_date) FROM tv_last3m").fetchone())

Last 3 month_partitions: ['202601', '202512', '202511']
Rows in tv_last3m: 30507
Date range last3m: (datetime.date(2025, 11, 1), datetime.date(2026, 1, 31))


9. SQL correcness - ground truth

In [30]:
# 1) Top 5 channels by AVG daily minutes (last 3 months)
gt_top5_avg_minutes = con.execute("""
    SELECT
        channelname,
        SUM(daily_total_minute) AS total_minutes,
        COUNT(DISTINCT watch_date) AS days,
        SUM(daily_total_minute) / NULLIF(COUNT(DISTINCT watch_date), 0) AS avg_daily_minutes
    FROM tv_last3m
    GROUP BY channelname
    ORDER BY avg_daily_minutes DESC
    LIMIT 5
""").df()

# 2) Top 5 channels by AVG daily unique viewers (last 3 months)
gt_top5_avg_viewers = con.execute("""
    SELECT
        channelname,
        SUM(nr_unique_viewers) AS total_viewers,
        COUNT(DISTINCT watch_date) AS days,
        SUM(nr_unique_viewers) / NULLIF(COUNT(DISTINCT watch_date), 0) AS avg_daily_viewers
    FROM tv_last3m
    GROUP BY channelname
    ORDER BY avg_daily_viewers DESC
    LIMIT 5
""").df()

print("GT Top5 AVG daily minutes:")
display(gt_top5_avg_minutes)

print("\nGT Top5 AVG daily unique viewers:")
display(gt_top5_avg_viewers)

GT Top5 AVG daily minutes:


,channelname,total_minutes,days,avg_daily_minutes
0,HRT 1,3.235219e+09,92,3.516542e+07
1,Nova TV,2.928746e+09,92,3.183420e+07
2,RTL,2.151862e+09,92,2.338981e+07
3,HRT 2,1.220909e+09,92,1.327076e+07
4,RTL 2,6.465007e+08,92,7.027182e+06



GT Top5 AVG daily unique viewers:


,channelname,total_viewers,days,avg_daily_viewers
0,HRT 1,17337179.0,92,188447.597826
1,Nova TV,16642025.0,92,180891.576087
2,RTL,16149675.0,92,175539.945652
3,HRT 2,12194092.0,92,132544.478261
4,RTL 2,7194845.0,92,78204.836957


10. Agent_sql

In [35]:
from app.agent import TOOLS, SYSTEM_PROMPT
from langchain.agents import create_agent

agent_sql = create_agent(
    model="openai:gpt-5-mini",
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

print("agent_sql tools:")
for t in TOOLS:
    print("-", t.name)

agent_sql tools:
- tv_schema
- tv_sql
- tv_rag_search
- web_search


11. SQL correctness - agent_sql predictions

In [36]:
debug_prompt = """
Use tv_sql to compute the top 3 channelname values by AVG DAILY minutes over the last 3 months.
Return only the 3 channel names, one per line. No extra text.
""".strip()

resp = agent_sql.invoke({"messages": [{"role": "user", "content": debug_prompt}]})

if isinstance(resp, dict) and "messages" in resp and resp["messages"]:
    print(resp["messages"][-1].content)
else:
    print(getattr(resp, "content", str(resp)))

HRT 1
Nova TV
RTL


In [37]:
import json
import re

def _extract_json(text: str) -> dict:
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError(f"No JSON object found. Raw response:\n{text[:500]}")
    return json.loads(m.group(0))

def ask_agent_sql_json(prompt: str) -> dict:
    result = agent_sql.invoke({"messages": [{"role": "user", "content": prompt}]})
    if isinstance(result, dict) and "messages" in result and result["messages"]:
        content = result["messages"][-1].content
    else:
        content = getattr(result, "content", str(result))
    return _extract_json(content)

prompt_minutes_force = """
You MUST use tv_sql to compute the answer.

Compute Top 5 channels by AVG DAILY viewing minutes over the LAST 3 MONTHS in the dataset.

Definition:
- total_minutes = SUM(daily_total_minute) over last 3 months
- days = COUNT(DISTINCT watch_date) over last 3 months
- avg_daily_minutes = total_minutes / days

Return ONLY valid JSON with this exact schema (rows must have exactly 5 items):
{
  "metric": "avg_daily_minutes",
  "window_months": 3,
  "top_k": 5,
  "rows": [
    {"channelname": "string", "value": number}
  ]
}
""".strip()

prompt_viewers_force = """
You MUST use tv_sql to compute the answer.

Compute Top 5 channels by AVG DAILY unique viewers over the LAST 3 MONTHS in the dataset.

Definition:
- total_viewers = SUM(nr_unique_viewers) over last 3 months
- days = COUNT(DISTINCT watch_date) over last 3 months
- avg_daily_viewers = total_viewers / days

Return ONLY valid JSON with this exact schema (rows must have exactly 5 items):
{
  "metric": "avg_daily_viewers",
  "window_months": 3,
  "top_k": 5,
  "rows": [
    {"channelname": "string", "value": number}
  ]
}
""".strip()

pred_minutes = ask_agent_sql_json(prompt_minutes_force)
pred_viewers = ask_agent_sql_json(prompt_viewers_force)

print("Pred minutes rows:", pred_minutes.get("rows", []))
print("Pred viewers rows:", pred_viewers.get("rows", []))

print("\nPred minutes JSON:\n", json.dumps(pred_minutes, ensure_ascii=False, indent=2))
print("\nPred viewers JSON:\n", json.dumps(pred_viewers, ensure_ascii=False, indent=2))

Pred minutes rows: [{'channelname': 'HRT 1', 'value': 35165423.89}, {'channelname': 'Nova TV', 'value': 31834198.37}, {'channelname': 'RTL', 'value': 23389808.01}, {'channelname': 'HRT 2', 'value': 13270755.34}, {'channelname': 'RTL 2', 'value': 7027181.69}]
Pred viewers rows: [{'channelname': 'HRT 1', 'value': 188447.5978260869}, {'channelname': 'Nova TV', 'value': 180891.5760869565}, {'channelname': 'RTL', 'value': 175539.9456521739}, {'channelname': 'HRT 2', 'value': 132544.4782608696}, {'channelname': 'RTL 2', 'value': 78204.8369565217}]

Pred minutes JSON:
 {
  "metric": "avg_daily_minutes",
  "window_months": 3,
  "top_k": 5,
  "rows": [
    {
      "channelname": "HRT 1",
      "value": 35165423.89
    },
    {
      "channelname": "Nova TV",
      "value": 31834198.37
    },
    {
      "channelname": "RTL",
      "value": 23389808.01
    },
    {
      "channelname": "HRT 2",
      "value": 13270755.34
    },
    {
      "channelname": "RTL 2",
      "value": 7027181.69
    }


12. SQL correcness scoring

In [38]:
import pandas as pd
import numpy as np

def topk_overlap(gt_channels, pred_channels, k=5):
    gt_set = set(gt_channels[:k])
    pred_set = set(pred_channels[:k])
    return len(gt_set & pred_set) / k

def exact_rank_match(gt_channels, pred_channels, k=5):
    return list(gt_channels[:k]) == list(pred_channels[:k])

def mape(gt_vals, pred_vals, eps=1e-9):
    gt_vals = np.array(gt_vals, dtype=float)
    pred_vals = np.array(pred_vals, dtype=float)
    denom = np.maximum(np.abs(gt_vals), eps)
    return float(np.mean(np.abs(pred_vals - gt_vals) / denom))

def build_compare_df(gt_df, pred_rows, gt_value_col, pred_value_key="value"):
    pred_df = pd.DataFrame(pred_rows).rename(columns={"value": "pred_value"})
    # gt_df: channelname + gt_value_col
    out = gt_df[["channelname", gt_value_col]].rename(columns={gt_value_col: "gt_value"}).merge(
        pred_df[["channelname", "pred_value"]],
        on="channelname",
        how="left"
    )
    out["abs_error"] = (out["pred_value"] - out["gt_value"]).abs()
    out["rel_error"] = out["abs_error"] / out["gt_value"].replace(0, np.nan)
    return out

# --- Ground truth channels + values ---
gt_minutes_channels = gt_top5_avg_minutes["channelname"].tolist()
gt_viewers_channels = gt_top5_avg_viewers["channelname"].tolist()

# --- Pred channels + values ---
pred_minutes_channels = [r["channelname"] for r in pred_minutes.get("rows", [])]
pred_viewers_channels = [r["channelname"] for r in pred_viewers.get("rows", [])]

# Overlap / rank
minutes_overlap5 = topk_overlap(gt_minutes_channels, pred_minutes_channels, k=5)
viewers_overlap5 = topk_overlap(gt_viewers_channels, pred_viewers_channels, k=5)

minutes_rank_ok = exact_rank_match(gt_minutes_channels, pred_minutes_channels, k=5)
viewers_rank_ok = exact_rank_match(gt_viewers_channels, pred_viewers_channels, k=5)

# Compare tables
cmp_minutes = build_compare_df(
    gt_top5_avg_minutes,
    pred_minutes["rows"],
    gt_value_col="avg_daily_minutes"
)

cmp_viewers = build_compare_df(
    gt_top5_avg_viewers,
    pred_viewers["rows"],
    gt_value_col="avg_daily_viewers"
)

# MAPE on matched rows (pred_value not null)
mape_minutes = mape(
    cmp_minutes["gt_value"].dropna(),
    cmp_minutes["pred_value"].dropna()
)
mape_viewers = mape(
    cmp_viewers["gt_value"].dropna(),
    cmp_viewers["pred_value"].dropna()
)

# Summary table
sql_correctness_summary = pd.DataFrame([
    {
        "query": "top5_avg_daily_minutes_last3m",
        "overlap@5": minutes_overlap5,
        "exact_rank_match": minutes_rank_ok,
        "mape": mape_minutes,
    },
    {
        "query": "top5_avg_daily_viewers_last3m",
        "overlap@5": viewers_overlap5,
        "exact_rank_match": viewers_rank_ok,
        "mape": mape_viewers,
    },
])

print("SQL correctness summary:")
display(sql_correctness_summary)

print("\nCompare (minutes):")
display(cmp_minutes)

print("\nCompare (viewers):")
display(cmp_viewers)

SQL correctness summary:


,query,overlap@5,exact_rank_match,mape
0,top5_avg_daily_minutes_last3m,1.0,True,1.342732e-10
1,top5_avg_daily_viewers_last3m,1.0,True,2.504954e-16



Compare (minutes):


,channelname,gt_value,pred_value,abs_error,rel_error
0,HRT 1,3.516542e+07,35165423.89,0.000543,1.545524e-11
1,Nova TV,3.183420e+07,31834198.37,0.002609,8.194581e-11
2,RTL,2.338981e+07,23389808.01,0.000761,3.253006e-11
3,HRT 2,1.327076e+07,13270755.34,0.003696,2.784806e-10
4,RTL 2,7.027182e+06,7027181.69,0.001848,2.629545e-10



Compare (viewers):


,channelname,gt_value,pred_value,abs_error,rel_error
0,HRT 1,188447.597826,188447.597826,5.820766e-11,3.088798e-16
1,Nova TV,180891.576087,180891.576087,0.000000e+00,0.000000e+00
2,RTL,175539.945652,175539.945652,2.910383e-11,1.657961e-16
3,HRT 2,132544.478261,132544.478261,2.910383e-11,2.195778e-16
4,RTL 2,78204.836957,78204.836957,4.365575e-11,5.582231e-16
